In [ ]:
# ============================================================
# ETAPA: Limpieza de Datos
# ============================================================


import pandas as pd          # Manipulación de datos
import numpy as np           # Operaciones numéricas

df = pd.read_csv("../data/telco_churn_raw.csv")

In [31]:
duplicados = df.duplicated().sum()
print(f"Filas duplicadas: {duplicados}")
# Si hubiera duplicados: df = df.drop_duplicates()

Filas duplicadas: 0


In [32]:
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100

missing_df = pd.DataFrame({
    "Valores Faltantes": missing,
    "Porcentaje (%):": missing_pct.round(2)
}).sort_values("Valores Faltantes", ascending=False)

print("=== VALORES FALTANTES POR COLUMNA ===")
print(missing_df[missing_df["Valores Faltantes"] > 0])

=== VALORES FALTANTES POR COLUMNA ===
Empty DataFrame
Columns: [Valores Faltantes, Porcentaje (%):]
Index: []


In [33]:
# TotalCharges viene como string
# Esta columna debería ser numérica pero pandas la lee como texto (object)
# Porque tiene espacios en blanco donde debería haber 0

print("Tipo de dato de Totalcharges:", df["TotalCharges"].dtype)
print("\nEjemplo de valores problemáticos en TotalCharges:")
# df[df["TotalCharges"] == " "].head()
print(df[df["TotalCharges"] == " "][["customerID","TotalCharges"]].head())

Tipo de dato de Totalcharges: object

Ejemplo de valores problemáticos en TotalCharges:
      customerID TotalCharges
488   4472-LVYGI             
753   3115-CZMZD             
936   5709-LVOEQ             
1082  4367-NUYAO             
1340  1371-DWPAZ             


In [ ]:
# Corrección de TotalCharges

df["TotalCharges"] = df["TotalCharges"].replace(" ", np.nan)

df["TotalCharges"] = pd.to_numeric(df["TotalCharges"])

nan_total = df["TotalCharges"].isnull().sum()
print(f"Valores Nan en TotalCharges: {nan_total}")

print("\nRegistros con TotalCharges vacío:")
df[df["TotalCharges"].isnull()][["tenure", "MonthlyCharges","TotalCharges"]].head(10)

# INTERPRETACIÓN: Los NaN corresponden a clientes con tenure=0
# Son clientes nuevos que AÚN no tienen cargos totales
# Decisión: imputar con 0 (no con la media, porque sería incorrecto conceptualmente)

df['TotalCharges'] = df['TotalCharges'].fillna(0)
print("\n✅ TotalCharges corregida correctamente")


Valores Nan en TotalCharges: 11

Registros con TotalCharges vacío:

✅ TotalCharges corregida correctamente


In [40]:
# Convertir variable objetivo a binaria
# Los modelos futuros necesitan 0/1 en lugar de "Yes"/"No"
# También facilita cálculos estadísticos

df["Churn_Binary"] = df["Churn"].map({"Yes": 1, "No": 0})

print("Verificación de conversión:")
print(df[["Churn", "Churn_Binary"]].value_counts())

Verificación de conversión:
Churn  Churn_Binary
No     0               5174
Yes    1               1869
Name: count, dtype: int64


In [ ]:
# Resumen de los tipos de variables

print("=== TIPOS DE VARIABLES EN EL DATASET ===\n")

numericas = df.select_dtypes(include=[np.number]).columns.tolist()
categoricas = df.select_dtypes(include=["object"]).columns.tolist()

print(f"Variables numéricas: ({len(numericas)}):")
for col in numericas:
    print(f" - {col}")

print(f"\nVariables categóricas: ({len(categoricas)}):")
for col in categoricas:
    print(f" - {col} | Valores únicos: {df[col].nunique()}")



=== TIPOS DE VARIABLES EN EL DATASET ===

Variables numéricas: (5):
 - SeniorCitizen
 - tenure
 - MonthlyCharges
 - TotalCharges
 - Churn_Binary

Variables categóricas: (17):
 - customerID | Valores únicos: 7043
 - gender | Valores únicos: 2
 - Partner | Valores únicos: 2
 - Dependents | Valores únicos: 2
 - PhoneService | Valores únicos: 2
 - MultipleLines | Valores únicos: 3
 - InternetService | Valores únicos: 3
 - OnlineSecurity | Valores únicos: 3
 - OnlineBackup | Valores únicos: 3
 - DeviceProtection | Valores únicos: 3
 - TechSupport | Valores únicos: 3
 - StreamingTV | Valores únicos: 3
 - StreamingMovies | Valores únicos: 3
 - Contract | Valores únicos: 3
 - PaperlessBilling | Valores únicos: 2
 - PaymentMethod | Valores únicos: 4
 - Churn | Valores únicos: 2


In [ ]:
# ---  Guardar dataset limpio ---

df.to_csv("../data/telco_churn_clean.csv", index=False)
print("\n✅ Dataset limpio guardado como 'telco_churn_clean.csv'")
print(f"   Dimensiones del dataset limpio: {df.shape[0]} filas x {df.shape[1]} columnas")


✅ Dataset limpio guardado como 'telco_churn_clean.csv'
   Dimensiones del dataset limpio: 7043 filas x 22 columnas



### 🛠️ Resultado de la Limpieza y Preprocesamiento

---

#### **Calidad de los Datos**
* **Duplicados:** 0 (Dataset íntegro).
* **Registros Totales:** 7,043 clientes (Sin pérdida de filas tras la limpieza).
* **Imputación:** Se corrigieron **11 valores** en `TotalCharges` (imputados con 0 para mantener la coherencia con clientes de antigüedad cero).

#### **Transformaciones de Datos**
* **Variable Objetivo:** `Churn` convertida a **binaria (0/1)** para facilitar el modelado predictivo.
* **Estructura Final del Dataset:**
    * **Variables Numéricas (5):** `tenure`, `MonthlyCharges`, `TotalCharges`, `SeniorCitizen`, `Churn_Binary`.
    * **Variables Categóricas (18):** Incluyen género, tipos de contrato, servicios contratados, métodos de pago, entre otras.

> **Estado:** Dataset listo para análisis exploratorio profundo (EDA).